In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

# Import sc, sn, sp data

In [ ]:
sc_adata = scv.read('/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/nuc_big_analysis/Liver_sc_big.h5ad')
sc_adata

In [ ]:
sc_adata.obs['disease'].value_counts()

In [ ]:
masld_list = ['healthy','NAFLD']
sc_adata = sc_adata[sc_adata.obs['disease'].isin(masld_list)].copy()

In [ ]:
sn_adata = scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/sn_MASLD_annotation.h5ad')
sn_adata

In [ ]:
sp_adata = scv.read('/home/jaehyunchoi/MASLD_collaboration/Spatial_data/Filtering_merged_adata.h5ad')
sp_adata

In [ ]:
sc_adata.raw.to_adata()
sc_adata

In [ ]:
sn_adata

In [ ]:
sp_adata

In [ ]:
adata_scrna = sc.read("/home/jaehyunchoi/MASLD_collaboration/MP_analysis/multiomics_analysis/pre/MASLD_scrna01.h5ad")
adata_snrna = sc.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102.h5ad')
adata_spatial = sc.read("/home/jaehyunchoi/MASLD_collaboration/MP_analysis/multiomics_analysis/pre/adata_spatial.h5ad")

In [ ]:
adata_scrna.obs['n_counts'] = adata_scrna.X.sum(axis=1).A1
adata_scrna.obs['n_genes'] = (adata_scrna.X > 0).sum(axis=1).A1
adata_snrna.obs['n_counts'] = adata_snrna.X.sum(axis=1).A1
adata_snrna.obs['n_genes'] = (adata_snrna.X > 0).sum(axis=1).A1
adata_spatial.obs['n_counts'] = adata_spatial.X.sum(axis=1).A1
adata_spatial.obs['n_genes'] = (adata_spatial.X > 0).sum(axis=1).A1

In [ ]:
adata_scrna

In [ ]:
sc_adata

In [ ]:
adata_scrna.obs['mt_percent'] = sc_adata.obs['percent.mito']
adata_scrna.obs['sample'] = sc_adata.obs['sample']
adata_scrna.obs['state'] = sc_adata.obs['disease']

In [ ]:
adata_scrna.obs['state'] 

In [ ]:
adata_snrna

In [ ]:
sn_adata

In [ ]:
adata_snrna.obs['mt_percent'] = adata_snrna.obs['mt_frac']
adata_snrna.obs['state'] = adata_snrna.obs['batch']

In [ ]:
adata_snrna

In [ ]:
adata_spatial

In [ ]:
sp_adata

In [ ]:
adata_spatial.obs['fov'] = sp_adata.obs['fov']
adata_spatial.obs['state'] = sp_adata.obs['disease']
adata_spatial.obs['slide'] = sp_adata.obs['slide']

In [ ]:
adata_spatial

In [ ]:
adata_spatial.obs['fov']

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")
adata_snrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")
adata_spatial.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Cosmx_v1/spatial_raw.h5ad")

# adata_scRNA analysis

In [ ]:
adata_scrna

In [ ]:
adata_scrna.obs['state']

In [ ]:
sc.pp.normalize_total(adata_scrna, target_sum=1e4)
sc.pp.log1p(adata_scrna)
sc.pp.highly_variable_genes(adata_scrna, flavor='seurat', n_top_genes=2000)

In [ ]:
# Scale data
sc.pp.scale(adata_scrna, max_value=10)
# Run PCA
sc.tl.pca(adata_scrna, svd_solver='arpack')
print(adata_scrna.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_scrna, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_scrna.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_scrna, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_scrna, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_scrna)
sc.pl.umap(adata_scrna, color=['sample'])

In [ ]:
adata_scrna

In [ ]:
sc.pl.umap(adata_scrna, color=['state'])

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_scrna02.h5ad")

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_scrna.obsm['X_pca'],adata_scrna.obs, 'sample')

# Harmony 결과를 저장
adata_scrna.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_scrna, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_scrna)

In [ ]:
sc.pl.umap(adata_scrna, color=['state'])

In [ ]:
adata_scpre = sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/pre/MASLD_scrna01.h5ad")
adata_scpre.obs['n_counts'] = adata_scpre.X.sum(axis=1).A1
adata_scpre.obs['n_genes'] = (adata_scpre.X > 0).sum(axis=1).A1
adata_scpre.obs['mt_percent'] = sc_adata.obs['percent.mito']
adata_scpre.obs['sample'] = sc_adata.obs['sample']
adata_scpre.obs['state'] = sc_adata.obs['disease']

In [ ]:
sc.pp.normalize_total(adata_scpre, target_sum=1e4)
sc.pp.log1p(adata_scpre)

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(adata_scpre, model = 'Healthy_Human_Liver.pkl', majority_voting = True)

In [ ]:
adata_scpre = predictions.to_adata()

In [ ]:
adata_scpre

In [ ]:
adata_scrna.obs['celltypist_liver']= adata_scpre.obs['majority_voting']

In [ ]:
sc.pl.umap(adata_scrna, color=['celltypist_liver'])

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False,        # 이걸 False로 해야 plt.savefig()가 동작
    return_fig=True    # <- 중요: 그림 객체 반환
)

# 직접 저장
plt.savefig("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Figure/MASLD_scRNA_celltypist_Liver.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
adata_scrna.obs['celltypist_liver'].value_counts()

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_scrna03.h5ad")

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/MP_analysis/multiomics_analysis/data/MASLD_scrna03.h5ad")

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")

In [ ]:
sc_adata = scv.read('/home/jaehyunchoi/MASLD_collaboration/nuc_big_analysis/Liver_sc_big.h5ad')
sc_adata

In [ ]:
adata_scrna

In [ ]:
adata_scrna.obs['cellannot_detail_JHB'] = sc_adata.obs['cellannot_detail_MP_Endo_Mes']

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='cellannot_detail_JHB',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='SLC7A5'
)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata_scrna.obsm['X_pca']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 30)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, sil_scores, marker='o')
plt.title('Silhouette Score for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

In [ ]:
sc.tl.leiden(adata_scrna, resolution=0.1, key_added="6cluster")
sc.pl.umap(adata_scrna, color=['6cluster'])

In [ ]:
adata_scrna.obs['11cluster'] = adata_scrna.obs['6cluster']

In [ ]:
ct = pd.crosstab(adata_scrna.obs['cellannot_detail_JHB'],adata_scrna.obs['celltypist_liver'])
ct

In [ ]:
ct = pd.crosstab(adata_scrna.obs['cellannot_detail_JHB'],adata_scrna.obs['11cluster'])
ct

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))

sc.pl.violin(
    adata_scrna,
    keys="SLC7A5",
    groupby="11cluster",
    multi_panel=False,
    show=False,  # ← 여기서 False로 해야 아래에서 수동 조정 가능
    ax=ax
)

# X축 레이블 회전
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")

In [ ]:
# healthy- MASLD DEG analysis

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")

In [ ]:
sc.pp.normalize_total(adata_scraw, target_sum=1e4)
sc.pp.log1p(adata_scraw)

In [ ]:
sc.tl.rank_genes_groups(
    adata_scraw,
    groupby="state",        # 비교할 obs 열
    groups=["NAFLD"],     # 비교할 그룹
    reference="healthy",
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
deg_sc_df = sc.get.rank_genes_groups_df(adata_scraw, group="NAFLD")
print(deg_sc_df.head())

In [ ]:
deg_sc_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/DEG/scTotal_MASLD_DEG.csv")

In [ ]:
print(deg_sc_df['names'].unique()[:50])  # 상위 50개 이름 확인
print("SLC7A5" in deg_sc_df['names'].values)
print("JUND" in deg_sc_df['names'].values)

In [ ]:
for gene in genes_of_interest:
    row = deg_sc_df.loc[deg_sc_df['names'] == gene]
    print(gene, row[['logfoldchanges', '-log10(padj)']])

In [ ]:
deg_sc_df['-log10(padj)'] = -np.log10(deg_sc_df['pvals_adj'])
deg_sc_df['-log10(padj)'] = deg_sc_df['-log10(padj)'].replace([np.inf, -np.inf], 350)
#genes_of_interest = ['SLC7A5','JUND']

plt.figure(figsize=(7, 6))
plt.scatter(deg_sc_df['logfoldchanges'], deg_sc_df['-log10(padj)'], c="grey", s=10, alpha=0.7)

sig_up = (deg_sc_df['logfoldchanges'] > 1) & (deg_sc_df['pvals_adj'] < 0.05)
sig_down = (deg_sc_df['logfoldchanges'] < -1) & (deg_sc_df['pvals_adj'] < 0.05)

plt.scatter(deg_sc_df.loc[sig_up, 'logfoldchanges'], deg_sc_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")
plt.scatter(deg_sc_df.loc[sig_down, 'logfoldchanges'], deg_sc_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

#for gene in genes_of_interest:
#    if gene in deg_sc_df['names'].values:
#        x = deg_sc_df.loc[deg_sc_df['names'] == gene, 'logfoldchanges'].values[0]
#        y = deg_sc_df.loc[deg_sc_df['names'] == gene, '-log10(padj)'].values[0]
#        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: Disease vs Normal")
plt.legend()
plt.ylim(0, 350)  # ✅ y축 강제 범위
plt.tight_layout()
plt.show()

In [ ]:
adata_scraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad")
adata_scrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_01.h5ad")

# adata_snRNA anaysis

In [ ]:
adata_snrna

In [ ]:
adata_snrna.obs['state']

In [ ]:
adata_snrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_snrna01.h5ad")

In [ ]:
sc.pp.normalize_total(adata_snrna, target_sum=1e4)
sc.pp.log1p(adata_snrna)
sc.pp.highly_variable_genes(adata_snrna, flavor='seurat', n_top_genes=2000)

In [ ]:
# Scale data
sc.pp.scale(adata_snrna, max_value=10)
# Run PCA
sc.tl.pca(adata_snrna, svd_solver='arpack')
print(adata_snrna.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_snrna, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_snrna.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_snrna, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_snrna, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_snrna)
sc.pl.umap(adata_snrna, color=['sample'])

In [ ]:
sc.pl.umap(adata_snrna, color=['state'])

In [ ]:
adata_snrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_snrna02.h5ad")

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_snrna.obsm['X_pca'],adata_snrna.obs, 'sample')

# Harmony 결과를 저장
adata_snrna.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_snrna, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_snrna)

In [ ]:
sc.pl.umap(adata_snrna, color=['state'])

In [ ]:
adata_snpre = sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_snrna01.h5ad")

In [ ]:
sc.pp.normalize_total(adata_snpre, target_sum=1e4)
sc.pp.log1p(adata_snpre)

In [ ]:
adata_snpre.obs_names_make_unique()

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(adata_snpre, model = 'Healthy_Human_Liver.pkl', majority_voting = True)

adata_snpre = predictions.to_adata()
adata_snrna.obs['celltypist_liver']= adata_snpre.obs['majority_voting']

In [ ]:
sc.pl.umap(adata_snrna, color=['celltypist_liver'])

In [ ]:
sc.pl.umap(
    adata_snrna,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_snrna,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False,        # 이걸 False로 해야 plt.savefig()가 동작
    return_fig=True    # <- 중요: 그림 객체 반환
)

# 직접 저장
plt.savefig("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Figure/MASLD_snRNA_celltypist_Liver.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
adata_snrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_snrna03.h5ad")

In [ ]:
adata_snrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/MP_analysis/multiomics_analysis/data/MASLD_snrna03.h5ad")

In [ ]:
adata_snrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_01.h5ad")

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")
adata_snrna=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_adata_01.h5ad")

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata_snrna.obsm['X_pca']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 30)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, sil_scores, marker='o')
plt.title('Silhouette Score for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

In [ ]:
sc.tl.leiden(adata_snrna, resolution=0.05, key_added="7cluster")
sc.pl.umap(adata_snrna, color=['7cluster'])

In [ ]:
adata_snraw=sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/snRNA_v1/snRNA_raw.h5ad")

In [ ]:
sc.pp.normalize_total(adata_snraw, target_sum=1e4)
sc.pp.log1p(adata_snraw)

In [ ]:
adata_snraw.obs['state']

In [ ]:
sc.tl.rank_genes_groups(
    adata_snraw,
    groupby="state",        # 비교할 obs 열
    groups=["MASLD"],     # 비교할 그룹
    reference="normal",
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

deg_sn_df = sc.get.rank_genes_groups_df(adata_snraw, group="MASLD")
print(deg_sn_df.head())

deg_sn_df.to_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/DEG/snTotal_MASLD_DEG.csv")

In [ ]:
deg_sn_df['-log10(padj)'] = -np.log10(deg_sn_df['pvals_adj'])
deg_sn_df['-log10(padj)'] = deg_sn_df['-log10(padj)'].replace([np.inf, -np.inf], 350)
#genes_of_interest = ['SLC7A5','JUND']

plt.figure(figsize=(7, 6))
plt.scatter(deg_sn_df['logfoldchanges'], deg_sn_df['-log10(padj)'], c="grey", s=10, alpha=0.7)

sig_up = (deg_sn_df['logfoldchanges'] > 1) & (deg_sn_df['pvals_adj'] < 0.05)
sig_down = (deg_sn_df['logfoldchanges'] < -1) & (deg_sn_df['pvals_adj'] < 0.05)

plt.scatter(deg_sn_df.loc[sig_up, 'logfoldchanges'], deg_sn_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")
plt.scatter(deg_sn_df.loc[sig_down, 'logfoldchanges'], deg_sn_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

#for gene in genes_of_interest:
#    if gene in deg_sn_df['names'].values:
#        x = deg_sn_df.loc[deg_sn_df['names'] == gene, 'logfoldchanges'].values[0]
#        y = deg_sn_df.loc[deg_sn_df['names'] == gene, '-log10(padj)'].values[0]
#        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: Disease vs Normal")
plt.legend()
plt.ylim(0, 350)  # ✅ y축 강제 범위
plt.tight_layout()
plt.show()

# adata_spatial analysis

In [ ]:
adata_spatial

In [ ]:
adata_spatial.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_spatial01.h5ad")

In [ ]:
sc.pp.normalize_total(adata_spatial, target_sum=1e4)
sc.pp.log1p(adata_spatial)
sc.pp.highly_variable_genes(adata_spatial, flavor='seurat', n_top_genes=2000)

In [ ]:
adata_spatial.obs['fov']

In [ ]:
# Scale data
sc.pp.scale(adata_spatial, max_value=10)
# Run PCA
sc.tl.pca(adata_spatial, svd_solver='arpack')
print(adata_spatial.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_spatial, color='fov')

import matplotlib.pyplot as plt
var_ratio = adata_spatial.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_spatial, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_spatial, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_spatial)
sc.pl.umap(adata_spatial, color=['fov'])

In [ ]:
sc.pl.umap(adata_spatial, color=['state'])

In [ ]:
adata_spatial.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_spatial02.h5ad")

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_spatial.obsm['X_pca'],adata_spatial.obs, 'fov')

# Harmony 결과를 저장
adata_spatial.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_spatial, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_spatial)

In [ ]:
sc.pl.umap(adata_spatial, color=['state'])

In [ ]:
sc.pl.umap(adata_spatial, color=['fov'])

In [ ]:
adata_spatial_pre = sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_spatial01.h5ad")

In [ ]:
sc.pp.normalize_total(adata_spatial_pre, target_sum=1e4)
sc.pp.log1p(adata_spatial_pre)

In [ ]:
adata_spatial_pre.obs_names_make_unique()

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(adata_spatial_pre, model = 'Healthy_Human_Liver.pkl', majority_voting = True)

adata_spatial_pre = predictions.to_adata()
adata_spatial.obs['celltypist_liver']= adata_spatial_pre.obs['majority_voting']

In [ ]:
sc.pl.umap(adata_spatial, color=['celltypist_liver'])

In [ ]:
sc.pl.umap(
    adata_spatial,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    adata_spatial,
    color='celltypist_liver',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False,        # 이걸 False로 해야 plt.savefig()가 동작
    return_fig=True    # <- 중요: 그림 객체 반환
)

# 직접 저장
plt.savefig("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Figure/MASLD_cosmx_celltypist_Liver.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
adata_spatial.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/data/MASLD_spatial03.h5ad")

In [ ]:
adata_spatial=sc.read("/home/jaehyunchoi/MASLD_collaboration/MP_analysis/multiomics_analysis/data/MASLD_spatial03.h5ad")

In [ ]:
adata_spatial.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Cosmx_v1/cosmx_adata_01.h5ad")

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata_spatial.obsm['X_pca']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 30)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, sil_scores, marker='o')
plt.title('Silhouette Score for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

In [ ]:
sc.tl.leiden(adata_spatial, resolution=1.1, key_added="12cluster")
sc.pl.umap(adata_spatial, color=['12cluster'])

In [ ]:
sc.pl.umap(adata_spatial, color=['celltypist_liver'])

In [ ]:
ct = pd.crosstab(adata_spatial.obs['celltypist_liver'],adata_spatial.obs['12cluster'])
ct

In [ ]:
del adata_spatial.obs['9cluster']

In [ ]:
adata_spatial.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Cosmx_v1/cosmx_adata_01.h5ad")